# Data Visualization

Top-to-bottom visual exploration of the Mooloolaba wave-buoy dataset. Runs cheaply — no RNN / LSTM / TCN training — so "Run All" finishes in seconds.

Reach for this notebook when you want to *look at the data itself*; use `forecast_comparison.ipynb` for modelling experiments.

All plotting goes through the `viz` package, so the same functions will apply to future sources (other buoys, atmospheric grids). Section 9 shows the multi-source pattern for when a second dataset lands.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LinearRegression

import forecast as fc
import viz

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Load and prepare

The cleaned CSV comes from `python -m wave_data`. Short gaps are interpolated so downstream visualisations don't flash NaN. The raw (pre-imputation) frame is kept as `df_raw` for the missing-value plot later.


In [ ]:
df_raw = fc.load_data()
df = df_raw.interpolate(limit=48).bfill().ffill()
assert df.isna().sum().sum() == 0, "expected fully imputed frame"

print(f"{len(df):,} rows  |  {df.index.min().date()} → {df.index.max().date()}  |  tz={df.index.tz}")
df.describe().round(2)


## 2. Time-series overview

A single-channel plot first, then an overlay of two related channels using the multi-source API (same pattern you'd use for multi-buoy comparisons).


In [ ]:
viz.plot_series(
    df["hsig_m"],
    title="Significant wave height (hsig_m) — 2015 → 2025",
    ylabel="Hs (m)",
    source_label="mooloolaba",
    alpha=0.7,
)
plt.tight_layout()
plt.show()


### Multi-channel overlay (multi-source API)

`plot_multi_source` takes a `dict[str, Series | DataFrame]` keyed by source label. Here we use it to compare two channels from one buoy; the same call signature handles multi-buoy overlays once other sources are wired up.


In [ ]:
# One-month slice so the traces are readable.
slc = slice("2024-06-01", "2024-07-01")
viz.plot_multi_source(
    {
        "hsig_m (significant)": df.loc[slc, "hsig_m"],
        "hmax_m (max)":         df.loc[slc, "hmax_m"],
    },
    title="hsig_m vs hmax_m — June 2024",
    ylabel="wave height (m)",
)
plt.tight_layout()
plt.show()


## 3. Channel distributions

Raw per-channel histograms. Mostly a sanity check — spot suspicious tails, unrealistic sentinels that survived cleaning, etc.


In [ ]:
channels = ["hsig_m", "hmax_m", "tz_s", "tp_s", "peak_dir_deg", "sst_c"]
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, col in zip(axes.flat, channels):
    df[col].hist(bins=60, ax=ax, color="#4c72b0", alpha=0.85)
    ax.set(title=col, ylabel="")
    ax.grid(alpha=0.3)
fig.suptitle("Channel distributions (mooloolaba, post-imputation)", y=1.02)
plt.tight_layout()
plt.show()


## 4. Missing-value inspection

Raw (pre-imputation) NaN counts per channel. SST is historically the noisiest instrument on this buoy; expect it to dominate.


In [ ]:
missing = df_raw.isna().sum().sort_values(ascending=False)
ax = missing.plot(kind="barh", color="#dd8452", figsize=(9, 3.5))
ax.set(title="Missing readings per channel (raw, pre-imputation)", xlabel="NaN count")
ax.invert_yaxis()
plt.tight_layout()
plt.show()
missing


## 5. Autocorrelation

How quickly does hsig_m become uncorrelated with its past self? This is the signal ceiling any model has to work with at a given horizon.


In [ ]:
viz.autocorrelation_curve(
    df["hsig_m"],
    max_hours=96,
    highlight_hours=[12, 24],
    source_label="mooloolaba",
)
plt.tight_layout()
plt.show()

for h in (6, 12, 24, 48, 72):
    r = df["hsig_m"].autocorr(lag=h * 2)
    print(f"Autocorr @ {h:>2}h: {r:.3f}")


## 6. Feature × forecast-horizon correlation

For each channel, how strongly does its value *now* correlate with `hsig_m` at some horizon *ahead*? Channels with a flat near-zero row (looking at you, `tp_s`) contribute little linear signal.


In [ ]:
df_feats = fc.encode_circular(df)
_, xcorr = viz.feature_horizon_heatmap(
    df_feats,
    target_col="hsig_m",
    feature_cols=["hsig_m", "hmax_m", "tz_s", "tp_s",
                  "peak_dir_deg_sin", "peak_dir_deg_cos", "sst_c"],
    source_label="mooloolaba",
)
plt.tight_layout()
plt.show()
xcorr.round(3)


## 7. Lookback × forecast-horizon (`hsig_m` self-similarity)

For `hsig_m` alone: does looking further back help predict further ahead? The grid tells you how much independent information older observations carry at each horizon.


In [ ]:
_, lb_grid = viz.lookback_horizon_heatmap(df["hsig_m"], source_label="mooloolaba")
plt.tight_layout()
plt.show()
lb_grid.round(3)


## 8. Forecaster diagnostics (cheap models only)

Two fast models — persistence (instant) and linear regression (a few seconds on 150k rows) — exercise the `viz` residual-analysis plots without any GPU/NN training.

For the full model comparison (baselines, trees, RNN / GRU / LSTM / TCN), see `forecast_comparison.ipynb`.


In [ ]:
X = fc.add_lag_features(
    fc.add_time_features(fc.encode_circular(df)),
    columns=["hsig_m", "hmax_m", "tp_s", "tz_s"],
    lags=[1, 4, 24],
)
y = fc.make_target(df)
X_train, X_test, y_train, y_test = fc.chronological_split(X, y, test_frac=0.2)

persistence = fc.evaluate(
    fc.PersistenceForecaster(), X_train, y_train, X_test, y_test, name="persistence",
)
linreg = fc.evaluate(
    LinearRegression(), X_train, y_train, X_test, y_test,
    name="linear-regression", baseline_preds=persistence.predictions,
)
fc.compare([persistence, linreg])


In [ ]:
viz.rmse_bar(
    [persistence, linreg],
    title=f"Test-set RMSE — {fc.HORIZON_HOURS}h hsig_m forecast",
)
plt.tight_layout()
plt.show()


In [ ]:
best = min([persistence, linreg], key=lambda r: r.metrics["RMSE"])
pred = pd.Series(best.predictions, index=X_test.index, name="pred")

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
viz.residual_timeseries(y_test, pred, model_name=best.name, ax=axes[0])
_, grouped = viz.residual_by_bin(
    y_test, pred,
    bins=[0, 0.75, 1.25, 1.75, 2.5, 10.0],
    statistic="std",
    title="Residual std by observed hsig_m bin",
    ylabel="std of errors (m)",
    ax=axes[1],
)
plt.tight_layout()
plt.show()
grouped


## 9. Multi-source patterns (reference, no execution)

When a second source lands — another buoy, a BOM coastal-wind grid, a GFS reanalysis product — plug it in without touching `viz`. The recipe is:

```python
import viz

sources = {
    "mooloolaba": mool_df,
    "brisbane":   bris_df,
    # "bom_wind": bom_df,
}

# Overlay the same variable across sources.
viz.plot_multi_source(sources, column="hsig_m",
                      title="hsig_m — buoy comparison")

# How similar are the same measurements across sources?
# (Optionally lag the non-anchor sources if one is upstream of the other.)
viz.cross_source_correlation(
    {name: df["hsig_m"] for name, df in sources.items()},
    lag_hours=0,
)

# Run the feature × horizon heatmap per source, source_label flows into the title.
for name, df in sources.items():
    viz.feature_horizon_heatmap(df, target_col="hsig_m", source_label=name)
```

No output from this cell — it's purely a template for when the data arrives.
